# FrameWeave Hybrid v6.0 (V5.1) on Ovis2.5-9B

Colab training notebook (**A100 40 GB**). FrameWeave Hybrid v6.0 implements formulation **V5.1**. It does **not** append join tokens. It writes a gated RMS-calibrated residual into Ovis visual embeddings after the Visual Embedding Table, then frozen Qwen3-8B reads the same visual-atom slots.

$$
\widetilde x_{ip} = x_{ip} + m_{ip}\,\chi_i\,a_{ip}\,c_{ip}\,u_{ip}
$$

- Clean pretrained Ovis, **frozen**. Hybrid is trained **joint-from-scratch** (no V3/V4/V5 weights, no GlobalTimeFix init checkpoint).
- $W_o=0$ (and bias $0$) at init → **exact Vanilla**. $P_L$ is small-Xavier (`gain=0.01`); Vanilla still holds because $u=W_o h=0$.
- $K=1$ remains identity because $\chi_i=0$ (pair mask excludes $i=j$)
- Question-guided frame pooling + non-periodic time $(\tau,\rho)$. Time lives in **Global only**.
- Global directed relation: $Q_i=W_Q\bar s_i+U_Q q$, $K_{ij}=W_K\bar s_j+A_K\phi_{ij}$
- Local Q/K are **post-LayerNorm**. Local has **no** $A_t\phi_{ij}$.
- Backbone: [`ATH-MaaS/Ovis2.5-9B`](https://huggingface.co/ATH-MaaS/Ovis2.5-9B) (NaViT + Qwen3-8B, $d_L=4096$)
- Multi-image frames, **not** Ovis video packing. Native resolution → variable $P_i$, masked pooling.
- Thinking mode **off**. Frozen VT / VET / LLM. Vanilla = hook bypass.
- Loss: $\mathrm{logsumexp}(\mathcal L_{\mathrm{cand}},\mathcal L_{\mathrm{corr}})-\log 2$ on all samples. Vanilla scores are stopgrad. Warmup **10%**.
- Default train: `VARIANT="Hybrid"`, `TRAIN_STAGE="joint"`, `INIT_CKPT=""`. Ablations: `GlobalTimeFix` (local off), `HybridNoTime`, `HybridNoGlobal`.
- Train data: PerceptionTest MC from LLaVA-Video-178K (same controlled split as v5). HERBench is a separate eval notebook.
- Torch runs in `/content/ovis_env` (Python 3.11 + torch 2.4.0 + flash-attn 2.7). No DTR.

## Source code

Mở notebook này từ GitHub hoặc Colab. Run All clone [FrameWeave](https://github.com/nqMinh-GR/FrameWeave.git) vào `/content/FrameWeave`, rồi symlink `reljoin/` và `run_*.py` vào `/content`. Không upload file source.

## `RUN_MODE`

`unit` | `smoke` | `overfit` | `lite`

Overfit gate (32 samples): V5.1 loss drop ≥ 30%, train acc ≥ 90%, no residual/grad explosion. Do not run `lite` until overfit passes.


## 1. Configuration

**Input:** none · **Output:** `RUN_MODE`, pins, paths


In [ ]:
# 1. Configuration — change RUN_MODE / STAGE here
from __future__ import annotations
from pathlib import Path

RUN_MODE = "unit"  # unit | smoke | overfit | lite
TRAIN_STAGE = "joint"  # joint-from-scratch (alias A still accepted by CLI)
VARIANT = "Hybrid"  # default: Global+Local together. Also GlobalTimeFix, HybridNoTime, HybridNoGlobal
INIT_CKPT = ""  # empty for joint Hybrid; do not load a GlobalTimeFix checkpoint

ROOT = Path("/content")
assert ROOT.exists(), "This notebook targets Google Colab (/content)."

MODEL_ID = "ATH-MaaS/Ovis2.5-9B"
MODEL_REVISION = "5a1f9b995c98ae5b85a28ec89d88d74295be0ea7"
TRAIN_DATASET_ID = "lmms-lab/LLaVA-Video-178K"
TRAIN_DATASET_REVISION = "6d8c562dc26d70042a0d9704d1cae58c94b89098"

VENV = ROOT / "ovis_env"
PYTHON = VENV / "bin" / "python"
OUT_DIR = ROOT / "outputs_reljoin_v6_ovis"
MODEL_DIR = ROOT / "checkpoints" / "ovis2_5_9b"

print({
    "RUN_MODE": RUN_MODE,
    "TRAIN_STAGE": TRAIN_STAGE,
    "VARIANT": VARIANT,
    "INIT_CKPT": INIT_CKPT,
    "ROOT": str(ROOT),
    "OUT_DIR": str(OUT_DIR),
    "backbone": MODEL_ID,
    "method": "FrameWeave-Hybrid",
    "method_version": "6.0.0",
    "formulation": "5.1.0",
})


## 2. Environment installation (`ovis_env`)

Official Ovis2.5-9B pins: `torch==2.4.0`, `transformers==4.51.3`, `flash-attn==2.7.0.post2`.
Stop if flash-attn install fails — the ViT hard-imports it.


In [ ]:
# 2. Bootstrap isolated ovis_env (Python 3.11 + torch 2.4.0 + flash-attn 2.7)
from __future__ import annotations
import os, subprocess
from pathlib import Path

assert Path("/content").exists(), "This notebook targets Google Colab (/content)."
ROOT = Path("/content")
VENV = ROOT / "ovis_env"
VENV_BIN = VENV / "bin"
PYTHON = VENV_BIN / "python"
PIP = VENV_BIN / "pip"
LOG = ROOT / "outputs_reljoin_v6_ovis" / "logs" / "cell4_env.log"
LOG.parent.mkdir(parents=True, exist_ok=True)
LOG.write_text("", encoding="utf-8")
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

def sh(cmd, check=True, env=None):
    line = "$ " + " ".join(map(str, cmd))
    print(line, flush=True)
    with LOG.open("a", encoding="utf-8") as f:
        f.write(line + "\n")
        proc = subprocess.run(
            list(map(str, cmd)),
            check=False,
            text=True,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            env=env,
        )
        f.write(proc.stdout or "")
        f.write(f"\nexit={proc.returncode}\n\n")
    print(proc.stdout or "", end="")
    if check and proc.returncode != 0:
        print("LOG:", LOG)
        print((proc.stdout or "")[-4000:])
        raise RuntimeError(f"command failed ({proc.returncode}). See {LOG}")
    return proc

sh(["apt-get", "update", "-qq"], check=False)
sh(["apt-get", "install", "-y", "-qq", "python3.11", "python3.11-venv", "python3.11-dev", "ffmpeg"], check=False)
py311 = Path("/usr/bin/python3.11")
assert py311.exists(), "Không tìm thấy python3.11 — kiểm tra apt install."

need_recreate = True
if PYTHON.exists():
    ver = subprocess.check_output([str(PYTHON), "-V"], text=True).strip()
    print("Existing venv:", ver)
    need_recreate = "3.11" not in ver
if need_recreate:
    if VENV.exists():
        sh(["rm", "-rf", str(VENV)])
    sh([str(py311), "-m", "venv", "--copies", str(VENV)])
if not PIP.exists():
    sh([str(PYTHON), "-m", "ensurepip", "--upgrade"], check=False)
if not PIP.exists():
    get_pip = Path("/tmp/get-pip.py")
    sh(["curl", "-sSL", "https://bootstrap.pypa.io/get-pip.py", "-o", str(get_pip)])
    sh([str(PYTHON), str(get_pip)])
sh([str(PYTHON), "-m", "pip", "install", "-q", "--upgrade", "pip", "setuptools<70", "wheel"])
print("Python in venv:", subprocess.check_output([str(PYTHON), "-V"], text=True).strip())
sh([
    str(PYTHON), "-m", "pip", "install", "--disable-pip-version-check",
    "torch==2.4.0", "torchvision==0.19.0",
    "--index-url", "https://download.pytorch.org/whl/cu121",
])
sh([
    str(PYTHON), "-m", "pip", "install", "--disable-pip-version-check", "--upgrade-strategy", "only-if-needed",
    "transformers==4.51.3", "numpy==1.25.0", "pillow==10.3.0", "moviepy==1.0.3",
    "huggingface_hub", "hf-transfer", "datasets", "accelerate", "einops", "sentencepiece",
    "opencv-python-headless", "pandas", "scipy", "tqdm", "ninja", "packaging",
])
print("Installing flash-attn==2.7.0.post2 (required by Ovis ViT)...")
fa_env = os.environ.copy()
fa_env["MAX_JOBS"] = "4"
sh(
    [str(PYTHON), "-m", "pip", "install", "flash-attn==2.7.0.post2", "--no-build-isolation"],
    env=fa_env,
)
check = subprocess.check_output([
    str(PYTHON), "-c",
    "import torch, transformers, flash_attn; "
    "print(torch.__version__); print(transformers.__version__); print(flash_attn.__version__); "
    "print(torch.__file__); print(transformers.__file__)"
], text=True)
print(check)
for line in check.splitlines():
    if "/site-packages/" in line:
        assert "/content/ovis_env/" in line, f"Package leaked outside venv: {line}"
print("ovis_env ready:", PYTHON)
print("LOG:", LOG)


In [ ]:
# 2b. Extra pip (same slot as Video-LLaVA cell 5). Writes a log so a failure is inspectable.
from __future__ import annotations
import os, subprocess
from pathlib import Path

ROOT = Path("/content")
PYTHON = ROOT / "ovis_env" / "bin" / "python"
LOG_DIR = ROOT / "outputs_reljoin_v6_ovis" / "logs"
LOG_DIR.mkdir(parents=True, exist_ok=True)
LOG = LOG_DIR / "cell5_pip.log"

if not PYTHON.exists():
    msg = (
        f"Missing {PYTHON}. Run the ovis_env bootstrap cell first "
        "(section 2), then re-run this cell.\n"
    )
    LOG.write_text(msg, encoding="utf-8")
    print(msg)
    raise FileNotFoundError(msg)

cmd = [
    str(PYTHON), "-m", "pip", "install",
    "--disable-pip-version-check",
    "--upgrade-strategy", "only-if-needed",
    "scikit-learn==1.3.2",
    "matplotlib==3.8.4",
    "numpy==1.25.0",
]
print("$", " ".join(cmd))
proc = subprocess.run(cmd, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
log_text = f"$ {' '.join(cmd)}\nexit={proc.returncode}\n\n{proc.stdout or ''}"
LOG.write_text(log_text, encoding="utf-8")
print(proc.stdout or "")
print("LOG:", LOG)

persist = Path(os.environ.get("RELJOIN_PERSIST", "/content/drive/MyDrive/reljoin_v6_ovis_outputs"))
if persist.parent.exists():
    try:
        persist.mkdir(parents=True, exist_ok=True)
        dest = persist / "logs" / LOG.name
        dest.parent.mkdir(parents=True, exist_ok=True)
        dest.write_text(log_text, encoding="utf-8")
        print("Copied log →", dest)
    except OSError as exc:
        print("Persist copy skipped:", exc)

if proc.returncode != 0:
    raise RuntimeError(f"cell 5 pip failed ({proc.returncode}). See {LOG}")
print("cell 5 pip OK")


## 3. Reproducibility — paths, Drive


In [ ]:
# 3. Paths, Drive
from __future__ import annotations
import os
from pathlib import Path

ROOT = Path("/content")
VENV = ROOT / "ovis_env"
PYTHON = VENV / "bin" / "python"
MODEL_DIR = ROOT / "checkpoints" / "ovis2_5_9b"
OUT_DIR = ROOT / "outputs_reljoin_v6_ovis"

for p in [MODEL_DIR.parent, OUT_DIR]:
    p.mkdir(parents=True, exist_ok=True)

try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    RELJOIN_PERSIST = Path("/content/drive/MyDrive/reljoin_v6_ovis_outputs")
except Exception as exc:
    print("Drive mount skipped:", exc)
    RELJOIN_PERSIST = OUT_DIR
RELJOIN_PERSIST.mkdir(parents=True, exist_ok=True)
os.environ["RELJOIN_PERSIST"] = str(RELJOIN_PERSIST)
os.environ["RELJOIN_ROOT"] = str(ROOT)

print({
    "RELJOIN_PERSIST": str(RELJOIN_PERSIST),
    "OUT_DIR": str(OUT_DIR),
    "MODEL_DIR": str(MODEL_DIR),
})


## 4. Clone source and verify


In [ ]:
# Fetch FrameWeave source. Run All clones the repo; do not upload .py files.
from __future__ import annotations

import subprocess
from pathlib import Path

REPO_URL = "https://github.com/nqMinh-GR/FrameWeave.git"
REPO_DIR = Path("/content/FrameWeave")
ROOT = Path("/content")
assert ROOT.exists(), "This notebook targets Google Colab (/content)."

if not (REPO_DIR / ".git").exists():
    subprocess.check_call(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)])
else:
    subprocess.check_call(["git", "-C", str(REPO_DIR), "pull", "--ff-only"])


def link(src: Path, dest: Path) -> None:
    src = src.resolve()
    if dest.is_symlink():
        if dest.resolve() != src:
            dest.unlink()
            dest.symlink_to(src)
        return
    if dest.exists():
        raise RuntimeError(
            f"{dest} already exists and is not the repo link. Restart the runtime, then Run All."
        )
    dest.symlink_to(src)


CODE = REPO_DIR / "ovis"
link(CODE / "reljoin", ROOT / "reljoin")
for script in sorted(CODE.glob("run_*.py")):
    link(script, ROOT / script.name)
print("FrameWeave source linked from", CODE)


In [ ]:
# 4. Verify FrameWeave sources
from pathlib import Path
ROOT = Path("/content")
required = [
    ROOT / "run_reljoin.py",
    ROOT / "reljoin" / "__init__.py",
    ROOT / "reljoin" / "config.py",
    ROOT / "reljoin" / "reljoin_modules.py",
    ROOT / "reljoin" / "ovis_hooks.py",
    ROOT / "reljoin" / "unit_tests.py",
    ROOT / "reljoin" / "train.py",
    ROOT / "reljoin" / "data_perception.py",
    ROOT / "reljoin" / "frame_io.py",
    ROOT / "reljoin" / "losses.py",
    ROOT / "run_herbench_oracle_reljoin.py",
]
missing = [str(p) for p in required if not p.exists()]
assert not missing, "Missing files — clone or symlink failed:\n" + "\n".join(missing)
print("All FrameWeave Hybrid sources present.")
for p in required:
    print(f"  OK {p} ({p.stat().st_size} bytes)")


## 5. Dataset / model download (gated by `RUN_MODE`)


In [ ]:
# 5. Conditional downloads (no HERBench in this training notebook)
from __future__ import annotations
import os, subprocess
from pathlib import Path

ROOT = Path("/content")
PYTHON = ROOT / "ovis_env" / "bin" / "python"
assert PYTHON.exists()
os.environ["PATH"] = f"{ROOT / 'ovis_env' / 'bin'}:{os.environ.get('PATH','')}"
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

try:
    RUN_MODE
except NameError:
    RUN_MODE = "unit"

MODEL_DIR = ROOT / "checkpoints" / "ovis2_5_9b"
MODEL_HF_ID = "ATH-MaaS/Ovis2.5-9B"
MODEL_REVISION = "5a1f9b995c98ae5b85a28ec89d88d74295be0ea7"

need_model = RUN_MODE != "unit"
need_perception = RUN_MODE in {"smoke", "overfit", "lite"}
print({"need_model": need_model, "need_perception": need_perception})

if need_model:
    print("Downloading Ovis2.5-9B (~18 GB bf16)...")
    MODEL_DIR.parent.mkdir(parents=True, exist_ok=True)
    py = (
        "import os\n"
        "os.environ['HF_HUB_ENABLE_HF_TRANSFER']='1'\n"
        "from huggingface_hub import snapshot_download\n"
        f"snapshot_download(repo_id='{MODEL_HF_ID}', revision='{MODEL_REVISION}', "
        f"local_dir=r'{MODEL_DIR}', ignore_patterns=['*.msgpack','*.h5','flax_model*','tf_model*'])\n"
        "print('model OK')\n"
    )
    subprocess.run([str(PYTHON), "-c", py], check=True)
    assert (MODEL_DIR / "config.json").exists()

if need_perception:
    print("PerceptionTest shard fetched by run_reljoin.py on demand.")

print("Download step done for RUN_MODE=", RUN_MODE)


## 6. FrameWeave Hybrid v6.0 (V5.1) in `reljoin/`

Vanilla bypass writes nothing. LLM weights are frozen but the FrameWeave multimodal forward is **not** `no_grad`, so gradients reach $\widetilde X$ and the adapter.

Question vector $h_q$ is the last non-pad hidden of a frozen text-only Qwen3 pass (`stopgrad`). Directed relations use $M_{ij}=f_i f_j\mathbf 1[i\neq j]$. Time features $\phi_{ij}=[\delta\tau,|\delta\tau|,\delta\rho,|\delta\rho|]$ plus a zero-init absolute map $W_{\mathrm{abs}}[\tau;\rho]$ feed **Global only**. Local Q/K are LayerNorm-ed after context aggregation; there is no $A_t$. Residual hidden is additive (not a $\lambda$ mix); $P_L$ is small-Xavier while $W_o=0$ keeps exact Vanilla. $u=W_o h$ with RMS calibration $c=\mathrm{RMS}(x)/(\mathrm{RMS}(x)+\mathrm{RMS}(u)+\epsilon)$. $K=1$ is identity via $\chi_i=0$.

Loss gathers letter logits at the first answer position: Hybrid forward with grad, Vanilla hook-bypass under `no_grad`, then per-sample $\mathrm{logsumexp}(L_{\mathrm{cand}},L_{\mathrm{corr}})-\log 2$.

## 7. Mathematical unit tests


In [ ]:
# 7. Unit tests via ovis_env
from __future__ import annotations
import subprocess
from pathlib import Path

ROOT = Path("/content")
PYTHON = ROOT / "ovis_env" / "bin" / "python"
cmd = [str(PYTHON), str(ROOT / "run_reljoin.py"), "--run-mode", "unit", "--root", "/content"]
print("$", " ".join(cmd))
proc = subprocess.run(cmd, cwd="/content")
if proc.returncode != 0:
    raise RuntimeError(f"unit tests failed ({proc.returncode}). Scroll up for traceback.")


## 8. External PerceptionTest training

_External uniform-frame warm-start — not definitive multi-evidence supervision._

`0_30_s_perceptiontest_videos_1.tar.gz` is the **training video shard** from LLaVA-Video-178K. It is named PerceptionTest because that is the source corpus. It is **not** HERBench and **not** the experiment test set.

- Train / val / early stop: PerceptionTest MC + `videos_1.tar.gz` (~5 GB)
- Held-out test: HERBench, in the eval notebook only

HERBench videos must never enter this cell.

Default: **joint** `Hybrid` from scratch — Global and Local both on, one LR, warmup 10%, writes `Hybrid.pt`. No `INIT_CKPT`. Ablations: `GlobalTimeFix`, `HybridNoTime`, `HybridNoGlobal`.


In [ ]:
# 8. Train FrameWeave Hybrid (gated by RUN_MODE)
from __future__ import annotations
import os, subprocess
from pathlib import Path

ROOT = Path("/content")
PYTHON = ROOT / "ovis_env" / "bin" / "python"
LOG = ROOT / "outputs_reljoin_v6_ovis" / "logs" / f"train_{globals().get('RUN_MODE', 'unit')}.log"
LOG.parent.mkdir(parents=True, exist_ok=True)
try:
    RUN_MODE
except NameError:
    RUN_MODE = "unit"
try:
    TRAIN_STAGE
except NameError:
    TRAIN_STAGE = "joint"
try:
    VARIANT
except NameError:
    VARIANT = "Hybrid"
try:
    INIT_CKPT
except NameError:
    INIT_CKPT = ""

os.environ.setdefault("RELJOIN_PERSIST", "/content/drive/MyDrive/reljoin_v6_ovis_outputs")
os.environ.setdefault("RELJOIN_ROOT", "/content")

if RUN_MODE in {"smoke", "overfit", "lite"}:
    chk = subprocess.check_output(
        [str(PYTHON), "-c", "from reljoin.data_perception import DATA_PARSER_VERSION; print(DATA_PARSER_VERSION)"],
        cwd="/content",
        text=True,
    ).strip()
    print("DATA_PARSER_VERSION:", chk)
    assert chk == "perception_conversations_mc_v3", (
        f"Colab vẫn dùng parser cũ ({chk}). Restart runtime rồi Run All để pull repo."
    )

    cmd = [
        str(PYTHON),
        str(ROOT / "run_reljoin.py"),
        "--run-mode",
        RUN_MODE,
        "--root",
        "/content",
        "--variant",
        VARIANT,
        "--stage",
        TRAIN_STAGE,
    ]
    if INIT_CKPT:
        cmd += ["--init-ckpt", str(INIT_CKPT)]
    print("$", " ".join(cmd))
    print("Training videos: LLaVA-Video-178K / 0_30_s_perceptiontest_videos_1.tar.gz (warm-start, not HERBench)")
    with LOG.open("w", encoding="utf-8") as logf:
        proc = subprocess.Popen(
            cmd,
            cwd="/content",
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )
        assert proc.stdout is not None
        for line in proc.stdout:
            print(line, end="")
            logf.write(line)
            logf.flush()
        proc.wait()
    print("FULL LOG:", LOG)
    if proc.returncode != 0:
        text = LOG.read_text(encoding="utf-8", errors="replace")
        tail = "\n".join(text.splitlines()[-40:])
        raise RuntimeError(
            f"run_reljoin exited {proc.returncode}. Real error is the log tail "
            f"(CUDA / flash-attn / OOM / download), not HERBench.\n"
            f"videos_1.tar.gz = PerceptionTest TRAIN shard. HERBench is eval-only.\n"
            f"LOG={LOG}\n\n{tail}"
        )
else:
    print(f"Skip training for RUN_MODE={RUN_MODE}")


## 9. Loss curves and overfit go/no-go


In [ ]:
# 9. Plot train_metrics.csv + overfit gate
from __future__ import annotations
from pathlib import Path
import json

ROOT = Path("/content")
OUT = ROOT / "outputs_reljoin_v6_ovis"
csv_path = OUT / "logs" / "train_metrics.csv"

try:
    RUN_MODE
except NameError:
    RUN_MODE = "unit"

if not csv_path.exists():
    print("No train_metrics.csv yet (run smoke/overfit/lite).")
else:
    import pandas as pd
    import matplotlib.pyplot as plt

    df = pd.read_csv(csv_path)
    display(df.tail(20))

    nll_col = "val_nll" if "val_nll" in df.columns else "val_loss"
    residual_col = "residual_rms_mean" if "residual_rms_mean" in df.columns else "residual_rms"
    grad_col = "grad_norm_mean" if "grad_norm_mean" in df.columns else "grad_norm"
    loss_col = "loss_v5" if "loss_v5" in df.columns else ("loss_combined" if "loss_combined" in df.columns else "ans_nll")

    fig, axes = plt.subplots(1, 3, figsize=(14, 3.5))
    axes[0].plot(df["epoch"], df[loss_col], marker="o", label=loss_col)
    if "loss_cand" in df:
        axes[0].plot(df["epoch"], df["loss_cand"], marker=".", label="loss_cand")
    if "loss_corr" in df:
        axes[0].plot(df["epoch"], df["loss_corr"], marker=".", label="loss_corr")
    if "loss_keep" in df:
        axes[0].plot(df["epoch"], df["loss_keep"], marker=".", label="loss_keep")
    if nll_col in df and df[nll_col].notna().any():
        axes[0].plot(df["epoch"], df[nll_col], marker="x", label=nll_col)
    axes[0].set_title("V5.1 loss")
    axes[0].legend()
    axes[1].plot(df["epoch"], df["train_acc"], marker="o", label="train_acc")
    if "val_acc" in df and df["val_acc"].notna().any():
        axes[1].plot(df["epoch"], df["val_acc"], marker="x", label="val_acc")
    axes[1].set_title("letter-logit acc")
    axes[1].legend()
    if residual_col in df:
        axes[2].plot(df["epoch"], df[residual_col], marker="o", label=residual_col)
        if grad_col in df:
            axes[2].plot(df["epoch"], df[grad_col], marker="x", label=grad_col)
        axes[2].set_title("residual / grad")
        axes[2].legend()
    fig.tight_layout()
    fig.savefig(OUT / "logs" / "loss_curves.png", dpi=120)
    plt.show()

    extra = [
        ("gate_p50", "gate_p95"),
        ("global_entropy_mean", "local_frame_entropy_mean"),
        ("local_ratio_mean", "paired_wins"),
    ]
    if any(a in df.columns or b in df.columns for a, b in extra):
        fig2, axes2 = plt.subplots(1, 3, figsize=(14, 3.5))
        for ax, (a, b) in zip(axes2, extra):
            if a in df.columns:
                ax.plot(df["epoch"], df[a], marker="o", label=a)
            if b in df.columns:
                ax.plot(df["epoch"], df[b], marker="x", label=b)
            ax.set_title(" / ".join(x for x in (a, b) if x in df.columns) or "n/a")
            ax.legend()
        fig2.tight_layout()
        fig2.savefig(OUT / "logs" / "hybrid_stats.png", dpi=120)
        plt.show()

    first = float(df[loss_col].iloc[0])
    last = float(df[loss_col].iloc[-1])
    drop = (first - last) / first if first > 0 else 0.0
    acc = float(df["train_acc"].iloc[-1])
    gmax = float(df[grad_col].max()) if grad_col in df else float("nan")
    rmax = float(df[residual_col].max()) if residual_col in df else float("nan")
    explosion = (gmax > 1e3) or (rmax > 1e3)
    gate = {
        "loss_first": first,
        "loss_last": last,
        "combined_drop": drop,
        "train_acc_last": acc,
        "grad_norm_max": gmax,
        "residual_rms_max": rmax,
        "combined_drop_ge_30pct": drop >= 0.30,
        "train_acc_ge_90pct": acc >= 0.90,
        "no_explosion": not explosion,
    }
    gate["overfit_go"] = (
        RUN_MODE != "overfit"
        or (gate["combined_drop_ge_30pct"] and gate["train_acc_ge_90pct"] and gate["no_explosion"])
    )
    (OUT / "overfit_gate.json").write_text(json.dumps(gate, indent=2), encoding="utf-8")
    print(json.dumps(gate, indent=2))
    if RUN_MODE == "overfit" and not gate["overfit_go"]:
        print("NO-GO: do not start lite (448) until this gate passes.")
    elif RUN_MODE == "overfit":
        print("GO: overfit gate passed. You may set RUN_MODE='lite' and re-run training cells.")


## 10. Export artifacts


In [ ]:
# 10. Copy outputs to Drive persist dir + zip
from __future__ import annotations
import shutil
from pathlib import Path
from datetime import datetime

ROOT = Path("/content")
OUT = ROOT / "outputs_reljoin_v6_ovis"
persist = Path("/content/drive/MyDrive/reljoin_v6_ovis_outputs")
if not persist.exists():
    persist = OUT
persist.mkdir(parents=True, exist_ok=True)
if OUT.exists() and persist.resolve() != OUT.resolve():
    shutil.copytree(OUT, persist, dirs_exist_ok=True)
    print("Copied", OUT, "→", persist)

stamp = datetime.utcnow().strftime("%Y%m%d_%H%M%S")
zip_path = ROOT / f"reljoin_v6_ovis_{stamp}"
if OUT.exists():
    shutil.make_archive(str(zip_path), "zip", root_dir=OUT)
    print("Zip:", f"{zip_path}.zip")
print("Persist:", persist)
